# Phase 1 — Training

Training two models on the Towards Comprehensive dataset and comparing them:
- **A1**: text only (TF-IDF + XGBoost) — used as baseline
- **A4**: text + Peerness + Repetition features — the full model


## Setup

- 80/20 train/test split, stratified by `(User1, User2)` pairs so no user pair leaks across splits
- `scale_pos_weight` set to ~128 to handle the class imbalance (0.8% positive)
- Primary metrics: F1, precision, recall (not accuracy — useless with 99:1 imbalance)


## Run training


In [1]:
import sys
from pathlib import Path

ROOT = Path.cwd()
for candidate in [ROOT, *ROOT.parents]:
    if (candidate / "src").is_dir() and (candidate / "requirements.txt").is_file():
        ROOT = candidate
        break
else:
    raise RuntimeError(
        "Cannot find F21MP project root. Open the F21MP folder in Cursor/VS Code."
    )

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.phase1_training.models.train import train_phase1

results = train_phase1(test_size=0.2, random_state=42, save_outputs=True, verbose=True)

Total samples: 15501
Train: 13031 | Test: 2470
Train pairs: 6691
Test pairs:  1673
Train CBLabel counts:
CBLabel
0    12930
1      101
Name: count, dtype: int64
scale_pos_weight: 128.02

=== Baseline A1 (text-only) ===
Accuracy:  0.9660
Precision: 0.0000
Recall:    0.0000
F1:        0.0000
ROC-AUC:   0.5095
Confusion matrix: [[2386, 59], [25, 0]]

=== Full model A4 (text + peerness + repetition) ===
Accuracy:  0.9996
Precision: 0.9615
Recall:    1.0000
F1:        0.9804
ROC-AUC:   1.0000
Confusion matrix: [[2444, 1], [0, 25]]

Models saved to: /Users/sagarmamindla/Desktop/F21MP/outputs/models
Metrics saved to: /Users/sagarmamindla/Desktop/F21MP/outputs/results/phase1_test_metrics.json


## Results

A1 (text only) basically predicts everything as non-cyberbullying — F1 = 0.00.
A4 (full model) catches all 25 cyberbullying cases in the test set with just 1 false positive — F1 = 0.98.

This confirms the core hypothesis: peerness and repetition features are essential for detection on this dataset.
Without them, the model can't overcome the class imbalance.


## Verify saved metrics


In [2]:
import json
from pathlib import Path

metrics_path = ROOT / "outputs" / "results" / "phase1_test_metrics.json"
with open(metrics_path) as f:
    metrics = json.load(f)

for name, m in metrics.items():
    print(f"\n{name}:")
    print(f"  F1={m['f1']:.4f}  Precision={m['precision']:.4f}  Recall={m['recall']:.4f}  ROC-AUC={m.get('roc_auc')}")


baseline_a1:
  F1=0.0000  Precision=0.0000  Recall=0.0000  ROC-AUC=0.5094969325153375

full_a4:
  F1=0.9804  Precision=0.9615  Recall=1.0000  ROC-AUC=0.9999836400817996


Phase 1 done. Models saved to `outputs/models/`, metrics to `outputs/results/phase1_test_metrics.json`.
Next: run Phase 2 validation on Twitter/YouTube data.
